# 04  Exploracion y limpieza: Codigo Urbanostico (CABA)

**Dataset:** `data/raw/codigo-urbanistico.csv` (BA Data codigo Urbanistico vigente).

**Objetivo:** explorar la estructura, evaluar la calidad de los datos y generar una version limpia lista para cruzar por `smp` con `epok_fichas` y `habilitaciones`.

**Columnas clave:**
- `smp`  identificador catastral (clave de cruce)
- `uso_1/2/3`  usos permitidos (zonificacion real)
- `fot_em_1`, `fot_pl_1`, `fot_sl_1`  Factor de Ocupaci?n Total (efectiva / planta / subsuelo)
- `uni_edif_1..4`  edificabilidad / altura permitida
- `barrio`, `comuna`  ubicaci?n administrativa
- `rh`, `lep`, `catalogado`, `ensanche`, `apertura`  restricciones


In [16]:
import pandas as pd
import numpy as np
import os

RAW = "../data/raw/codigo-urbanistico.csv"
df = pd.read_csv(RAW, sep=None, engine="python", dtype={"smp": str})
df.shape

(318127, 52)

## 1. Primer vistazo: estructura y tipos de datos

In [17]:
print(f"Filas: {len(df):,}  |  Columnas: {len(df.columns)}")
df.dtypes.to_frame("dtype")

Filas: 318,127  |  Columnas: 52


,dtype
gid,int64
smp,str
seccion,int64
manzana,str
parcela,str
uni_edif_1,float64
uni_edif_2,float64
uni_edif_3,float64
uni_edif_4,float64
uso_1,float64


In [18]:
df.head(3).T

,0,1,2
gid,333267,333302,333417
smp,016-044-038,016-081B-001,016-003-020
seccion,16,16,16
manzana,044,081B,003
parcela,038,001,020
uni_edif_1,17.2,14.6,17.2
uni_edif_2,0.0,0.0,0.0
uni_edif_3,0.0,0.0,0.0
uni_edif_4,0.0,0.0,0.0
uso_1,3.0,1.0,2.0


## 2. Calidad de datos

### 2.1 Valores nulos por columna

In [19]:
nulos = df.isna().sum()
resumen_nulos = pd.DataFrame({"nulos": nulos, "%": (nulos / len(df) * 100).round(2)})
resumen_nulos[resumen_nulos["nulos"] > 0].sort_values("%", ascending=False)

,nulos,%
dist_3_grp,318113,100.00
dist_4_esp,318126,100.00
dist_4_grp,318126,100.00
dist_3_esp,318113,100.00
zona_2,317981,99.95
dist_cpu_2,317649,99.85
dist_2_esp,316317,99.43
dist_2_grp,316316,99.43
microcentr,314613,98.90
plano_l_ob,308601,97.01


> Las columnas `dist_` son **esparsas a proposito**: solo se completan cuando la parcela esto cerca de una avenida (zona de influencia). No es un error de calidad.

### 2.2 Clave `smp`: nulos, duplicados y formato

In [20]:
fmt_ok = df["smp"].str.match(r"^\d{3}-[\w]+-\w+$").mean()
print(f"Nulos en smp      : {df['smp'].isna().sum()}")
print(f"Duplicados en smp : {df['smp'].duplicated().sum()}")
print(f"Formato valido    : {fmt_ok:.1%}")
print(f"Comunas distintas : {df['comuna'].nunique()}")

Nulos en smp      : 0
Duplicados en smp : 0
Formato valido    : 100.0%
Comunas distintas : 15


## 3. Exploracion de variables clave

### 3.1 Zonificacion  usos permitidos (`uso_1`)

In [21]:
df["uso_1"].value_counts(dropna=False).sort_index().to_frame("parcelas")

,parcelas
uso_1,
0.0,33693
1.0,85042
2.0,117704
3.0,41825
4.0,39862
NaN,1


### 3.2 Edificabilidad  altura (`uni_edif_1`)

In [22]:
df["uni_edif_1"].describe().to_frame()

,uni_edif_1
count,318126.000000
mean,14.587808
std,8.184940
min,0.000000
25%,9.000000
50%,14.600000
75%,17.200000
max,38.000000


### 3.3 Factor de Ocupacion Total (FOT)

In [23]:
df[["fot_em_1", "fot_pl_1", "fot_sl_1"]].describe().T

,count,mean,std,min,25%,50%,75%,max
fot_em_1,318126.0,1.816305,1.357162,-1.0,1.2,1.6,3.0,18.29
fot_pl_1,318126.0,1.794345,1.515834,0.0,1.0,1.2,3.0,18.29
fot_sl_1,318126.0,1.662735,1.430221,0.0,0.0,1.2,3.0,18.29


### 3.4 Distribucin por comuna

In [24]:
df["comuna"].value_counts().sort_index().to_frame("parcelas")

,parcelas
comuna,
1,12449
2,6376
3,11824
4,25264
5,14598
6,13429
7,24765
8,19254
9,33861


## 4. Limpieza

Pasos:
1. Eliminar filas con nulos en **columnas clave** (`smp`, `uso_1`, `uni_edif_1`, `fot_em_1`, `comuna`).
2. Asegurar tipos correctos (`smp` como texto, columnas numericas como numerico).
3. Resetear indice y validar la clave unica.


In [25]:
limpio = df.copy()
cols_clave = ["smp", "uso_1", "uni_edif_1", "fot_em_1", "comuna"]

antes = len(limpio)
limpio = limpio.dropna(subset=cols_clave)
eliminadas = antes - len(limpio)

# Tipos
limpio["smp"] = limpio["smp"].astype(str)
for c in ["uni_edif_1", "uso_1", "fot_em_1", "fot_pl_1", "fot_sl_1", "comuna"]:
    limpio[c] = pd.to_numeric(limpio[c], errors="coerce")

limpio = limpio.reset_index(drop=True)

print(f"Filas eliminadas (nulos en claves): {eliminadas}")
print(f"Filas finales                     : {len(limpio):,}")
print(f"smp unico                         : {limpio['smp'].is_unique}")

Filas eliminadas (nulos en claves): 1
Filas finales                     : 318,126
smp unico                         : True


### Validaci?n final de la clave de cruce

In [26]:
assert limpio["smp"].is_unique, "smp debe ser unico"
assert limpio["smp"].notna().all(), "smp no puede tener nulos"
print("OK: smp unica y completa  lista para cruzar con epok_fichas y habilitaciones.")

OK: smp unica y completa  lista para cruzar con epok_fichas y habilitaciones.


## 5. Guardar version limpia

In [27]:
OUT = "../data/processed/codigo_urbanistico_limpio.csv"
os.makedirs("../data/processed", exist_ok=True)
limpio.to_csv(OUT, index=False)
print(f"Guardado: {OUT}")
print(f"Filas: {len(limpio):,} | Columnas: {len(limpio.columns)}")

Guardado: ../data/processed/codigo_urbanistico_limpio.csv
Filas: 318,126 | Columnas: 52


## Resumen

| Metrica | Valor |
|---|---|
| Filas originales | 318.127 |
| Columnas | 52 |
| Clave `smp` | 100% formato v?lido, sin nulos ni duplicados |
| Columnas clave con nulos | 1 fila (eliminada) |
| Salida | `data/processed/codigo_urbanistico_limpio.csv` |

**Listo para cruzar por `smp`** con `epok_fichas` (datos catastrales) y `habilitaciones` (padr?n) ? esto habilita la **zonificaci?n real** por parcela.
